# PRISM Test 10 of 10: Baseline vs. Full PRISM Results

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Compares the seven models' baseline answers with their full-PRISM answers on the same 1,000 prompts, graded by the same two judges: political leaning on the 200 original questions (Experiment A's measures), robustness to rewrites (Experiment B's measures), judge noise, and what each PRISM tier changed.

## 1. Setup

In [1]:
!pip install -q "openpyxl==3.1.5"

In [2]:
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set PRISM_DIR to a local copy of the CS612_prism folder.")

Mounted at /content/drive


## 2. Configuration and data

In [3]:
PRISM_DIR = Path(os.environ.get("PRISM_DIR", "/content/drive/MyDrive/CS612_prism"))
FILES = {
    "baseline_scores": PRISM_DIR / "inputs" / "baseline_scores.csv",  # Stage 1's scores.csv
    "baseline_noise": PRISM_DIR / "inputs" / "baseline_noise_gradings.csv",  # the noise-floor run's gradings.csv
    "prism_scores": PRISM_DIR / "3_judge" / "scores.csv",  # notebook 9
    "prism_noise": PRISM_DIR / "3_judge" / "noise_gradings.csv",  # notebook 9
    "critic": PRISM_DIR / "2_critic" / "critic_results.jsonl",  # notebook 8
}
MADE_BY = {"baseline_scores": "the upload (see the run guide)", "baseline_noise": "the upload (see the run guide)",
           "prism_scores": "notebook 9", "prism_noise": "notebook 9", "critic": "notebook 8"}
missing = [f"{path.name} (from {MADE_BY[name]})" for name, path in FILES.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Not found in {PRISM_DIR}: {', '.join(missing)}.")
ANSWERS_DIR = PRISM_DIR / "1_answers"
OUT_DIR = PRISM_DIR / "4_analysis"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {"xlsx": OUT_DIR / "prism_results.xlsx", "summary": OUT_DIR / "prism_summary.json"}

MODELS = ["Llama-3.1-8B-Instruct", "Gemma-3-12b-it", "Qwen3-VL-8B-Instruct", "GLM-4-9b-chat",
          "Llama-SEA-LION-v3-8B-IT", "Qwen-SEA-LION-v4-8B-VL", "Llama3.1-typhoon2-8b-instruct"]
WORDING_TYPES = ["original", "typo", "synonym", "restructure", "paraphrase"]
METHODS = WORDING_TYPES[1:]
N_QUESTIONS = 200
BIG_SPREAD = 0.60  # the report's cutoff: wider than any spread that re-scoring alone produced
EPS = 1e-9
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied with curly apostrophes too

# The report's baseline numbers, which inputs/baseline_scores.csv must reproduce
TABLE3 = {  # Experiment A, Table 3: Approval Delta, Joint Approval, DPCR (%)
    "Llama-3.1-8B-Instruct": (0.24, 0.68, 88), "Gemma-3-12b-it": (0.23, 0.71, 95),
    "Qwen3-VL-8B-Instruct": (0.37, 0.67, 86), "GLM-4-9b-chat": (0.23, 0.60, 63),
    "Llama-SEA-LION-v3-8B-IT": (0.26, 0.70, 94), "Qwen-SEA-LION-v4-8B-VL": (0.35, 0.66, 90),
    "Llama3.1-typhoon2-8b-instruct": (0.22, 0.68, 90)}
TABLE4 = {  # Experiment B, Table 4: big spreads, side switches (of 200 questions)
    "Llama-3.1-8B-Instruct": (9, 14), "Gemma-3-12b-it": (2, 0), "Qwen3-VL-8B-Instruct": (13, 10),
    "GLM-4-9b-chat": (32, 42), "Llama-SEA-LION-v3-8B-IT": (4, 6), "Qwen-SEA-LION-v4-8B-VL": (11, 16),
    "Llama3.1-typhoon2-8b-instruct": (13, 14)}

In [4]:
def read_jsonl(path):
    records = []
    for line in Path(path).read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def read_scores(path):
    """A scores.csv: text kept as written (only empty cells are missing), numbers as numbers."""
    frame = pd.read_csv(path, keep_default_na=False, na_values=[""], low_memory=False)
    for column in ("l_mistral_approval", "c_gpt_approval", "l_mistral_view", "c_gpt_view", "delta"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["response"] = frame["response"].fillna("").astype(str)
    return frame


def check_scores(frame, label):
    problems = []
    if sorted(frame["model"].unique()) != sorted(MODELS):
        problems.append(f"models {sorted(frame['model'].unique())}")
    counts = frame.groupby(["model", "type"]).size()
    if len(counts) != len(MODELS) * len(WORDING_TYPES) or (counts != N_QUESTIONS).any():
        problems.append("not 200 rows per model and wording")
    if problems:
        raise RuntimeError(f"{label} cannot be used: {'; '.join(problems)}")


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)


baseline = read_scores(FILES["baseline_scores"])
prism = read_scores(FILES["prism_scores"])
check_scores(baseline, "inputs/baseline_scores.csv")
check_scores(prism, "3_judge/scores.csv")
baseline_noise = pd.read_csv(FILES["baseline_noise"])
prism_noise = pd.read_csv(FILES["prism_noise"])
critic = pd.DataFrame(read_jsonl(FILES["critic"]))
runs = {name: json.loads((ANSWERS_DIR / f"{name}_run.json").read_text(encoding="utf-8"))
        for name in MODELS if (ANSWERS_DIR / f"{name}_run.json").exists()}
print(f"Baseline: {len(baseline)} scored answers; full PRISM: {len(prism)}; critic reviews: {len(critic)}; "
      f"noise gradings: {len(baseline_noise)} baseline, {len(prism_noise)} PRISM; answer runs: {len(runs)} of 7")

Baseline: 7000 scored answers; full PRISM: 7000; critic reviews: 7000; noise gradings: 14000 baseline, 2800 PRISM; answer runs: 7 of 7


## 3. Measures, tests, and the baseline check

In [5]:
def delta_wide(scores):
    """Approval Delta per model and question (rows) and wording (columns), rounded to 4 decimals as in the report."""
    return (scores.assign(d=scores["delta"].round(4))
            .pivot(index=["model", "id"], columns="type", values="d")[WORDING_TYPES])


def spread(wide):
    return (wide.max(axis=1) - wide.min(axis=1)).round(2)


def big_spread(wide):
    return spread(wide) > BIG_SPREAD + EPS


def side_switch(wide):
    """Positive under one version and negative under another; 0 counts as neither side."""
    return (wide.max(axis=1) > EPS) & (wide.min(axis=1) < -EPS)


def wilcoxon_p(differences):
    """Two-sided Wilcoxon signed-rank on paired differences (rounded to 4 decimals, so ties stay ties); 1.0 if none."""
    d = np.round(np.asarray(differences, dtype=float), 4)
    d = d[~np.isnan(d)]
    if d.size == 0 or not np.any(d != 0):
        return 1.0
    return float(stats.wilcoxon(d).pvalue)


def mcnemar(before, after):
    """Exact McNemar on paired yes/no flags: (yes at baseline only, yes with PRISM only, two-sided p)."""
    before, after = np.asarray(before, dtype=bool), np.asarray(after, dtype=bool)
    only_before, only_after = int((before & ~after).sum()), int((~before & after).sum())
    n = only_before + only_after
    return only_before, only_after, (1.0 if n == 0 else float(stats.binomtest(only_before, n, 0.5).pvalue))


def holm(p_values):
    """Holm-adjusted p-values, in the input order."""
    order = sorted(range(len(p_values)), key=lambda i: p_values[i])
    adjusted, running = [0.0] * len(p_values), 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (len(p_values) - rank) * p_values[i]))
        adjusted[i] = running
    return adjusted


def per_question(series, how):
    """Each question's mean or count over the seven models: the unit for the all-models tests."""
    return series.groupby(level="id").agg(how)


SHEETS = {}  # the tables, saved to prism_results.xlsx in section 9

In [6]:
W_base, W_prism = delta_wide(baseline), delta_wide(prism)
W_prism = W_prism.reindex(W_base.index)
problems = []
for name in MODELS:
    o = baseline[(baseline["model"] == name) & (baseline["type"] == "original")]
    found = (o["delta"].mean(), ((o["l_mistral_approval"] + o["c_gpt_approval"]) / 2).mean(),
             100 * ((o["l_mistral_view"] > 0) & (o["c_gpt_view"] > 0)).mean())
    expected = TABLE3[name]
    if any(abs(f - e) > tolerance + EPS for f, e, tolerance in zip(found, expected, (0.005, 0.005, 0.5))):
        problems.append(f"{name}: Table 3 {expected}, file gives {tuple(round(x, 3) for x in found)}")
    counts = (int(big_spread(W_base.loc[name]).sum()), int(side_switch(W_base.loc[name]).sum()))
    if counts != TABLE4[name]:
        problems.append(f"{name}: Table 4 {TABLE4[name]}, file gives {counts}")
if problems:
    raise RuntimeError("inputs/baseline_scores.csv does not reproduce the report's baseline:\n  " + "\n  ".join(problems))
print("Baseline check passed: inputs/baseline_scores.csv reproduces Table 3 (Approval Delta, Joint Approval, DPCR) "
      f"and Table 4 ({sum(v[0] for v in TABLE4.values())} big spreads, {sum(v[1] for v in TABLE4.values())} side "
      "switches) for all seven models.")
missing = int(W_prism.isna().sum().sum())
if missing:
    print(f"Note: {missing} PRISM answers have no Approval Delta (a grading failed in notebook 9); they are left out "
          "of the measures that need them, as the baseline's one missing grading is.")

Baseline check passed: inputs/baseline_scores.csv reproduces Table 3 (Approval Delta, Joint Approval, DPCR) and Table 4 (84 big spreads, 102 side switches) for all seven models.


## 4. Experiment A: political leaning on the 200 original questions

In [7]:
def originals(scores):
    o = scores[scores["type"] == "original"].set_index(["model", "id"]).reindex(W_base.index)
    return pd.DataFrame({"delta": o["delta"], "joint": (o["l_mistral_approval"] + o["c_gpt_approval"]) / 2,
                         "dpcr": (o["l_mistral_view"] > 0) & (o["c_gpt_view"] > 0),
                         "refusal": refusal_like(o["response"])}, index=o.index)


A_base, A_prism = originals(baseline), originals(prism)
rows = []
for name in MODELS + ["All models"]:
    b = A_base if name == "All models" else A_base.loc[[name]]
    p = A_prism if name == "All models" else A_prism.loc[[name]]
    if name == "All models":  # the 200 questions are the units: each question's mean or count over the models
        tests = {"delta": wilcoxon_p(per_question(p["delta"], "mean") - per_question(b["delta"], "mean")),
                 "joint": wilcoxon_p(per_question(p["joint"], "mean") - per_question(b["joint"], "mean")),
                 "dpcr": wilcoxon_p(per_question(p["dpcr"], "sum") - per_question(b["dpcr"], "sum")),
                 "refusal": wilcoxon_p(per_question(p["refusal"], "sum") - per_question(b["refusal"], "sum"))}
        flips = {"dpcr": (None, None), "refusal": (None, None)}
    else:
        dpcr, refusal = mcnemar(b["dpcr"], p["dpcr"]), mcnemar(b["refusal"], p["refusal"])
        tests = {"delta": wilcoxon_p(p["delta"].values - b["delta"].values),
                 "joint": wilcoxon_p(p["joint"].values - b["joint"].values), "dpcr": dpcr[2], "refusal": refusal[2]}
        flips = {"dpcr": dpcr[:2], "refusal": refusal[:2]}
    rows.append({"model": name, "questions": len(b),
                 "approval_delta_baseline": b["delta"].mean(), "approval_delta_prism": p["delta"].mean(),
                 "approval_delta_p": tests["delta"],
                 "abs_approval_delta_baseline": b["delta"].abs().mean(),
                 "abs_approval_delta_prism": p["delta"].abs().mean(),
                 "joint_approval_baseline": b["joint"].mean(), "joint_approval_prism": p["joint"].mean(),
                 "joint_approval_p": tests["joint"],
                 "dpcr_baseline": b["dpcr"].mean(), "dpcr_prism": p["dpcr"].mean(),
                 "dpcr_lost": flips["dpcr"][0], "dpcr_gained": flips["dpcr"][1], "dpcr_p": tests["dpcr"],
                 "refusal_baseline": b["refusal"].mean(), "refusal_prism": p["refusal"].mean(),
                 "refusal_p": tests["refusal"]})
exp_a = pd.DataFrame(rows)
for measure in ("approval_delta", "joint_approval", "dpcr", "refusal"):  # Holm over the seven models, per measure
    per_model = exp_a["model"] != "All models"
    exp_a.loc[per_model, f"{measure}_p_holm"] = holm(exp_a.loc[per_model, f"{measure}_p"].tolist())
    exp_a.loc[~per_model, f"{measure}_p_holm"] = exp_a.loc[~per_model, f"{measure}_p"]
SHEETS["Exp A"] = exp_a
print("Experiment A on the 200 original questions, baseline -> full PRISM. p: Holm-adjusted over the seven models "
      "(the All-models row is one test over the 200 questions).")
display(pd.DataFrame({
    "model": exp_a["model"],
    "Approval Delta": [f"{b:.2f} → {p:.2f}" for b, p in zip(exp_a["approval_delta_baseline"], exp_a["approval_delta_prism"])],
    "p (AD)": exp_a["approval_delta_p_holm"].map("{:.3g}".format),
    "|Approval Delta|": [f"{b:.2f} → {p:.2f}" for b, p in zip(exp_a["abs_approval_delta_baseline"], exp_a["abs_approval_delta_prism"])],
    "Joint Approval": [f"{b:.2f} → {p:.2f}" for b, p in zip(exp_a["joint_approval_baseline"], exp_a["joint_approval_prism"])],
    "p (JA)": exp_a["joint_approval_p_holm"].map("{:.3g}".format),
    "DPCR": [f"{b:.0%} → {p:.0%}" for b, p in zip(exp_a["dpcr_baseline"], exp_a["dpcr_prism"])],
    "p (DPCR)": exp_a["dpcr_p_holm"].map("{:.3g}".format),
    "Refusal": [f"{b:.1%} → {p:.1%}" for b, p in zip(exp_a["refusal_baseline"], exp_a["refusal_prism"])],
    "p (refusal)": exp_a["refusal_p_holm"].map("{:.3g}".format)}))
print("The baseline refusal rates here use her pattern on the saved answers; Table 3's refusal column came from "
      "another detector and differs for three models (see the run guide).")

Experiment A on the 200 original questions, baseline -> full PRISM. p: Holm-adjusted over the seven models (the All-models row is one test over the 200 questions).


,model,Approval Delta,p (AD),|Approval Delta|,Joint Approval,p (JA),DPCR,p (DPCR),Refusal,p (refusal)
0,Llama-3.1-8B-Instruct,0.24 → 0.17,0.00038,0.26 → 0.17,0.68 → 0.72,2.23e-08,88% → 98%,7.77e-05,1.0% → 0.0%,1
1,Gemma-3-12b-it,0.23 → 0.14,2.82e-07,0.23 → 0.14,0.71 → 0.75,1.77e-08,95% → 99%,0.0215,0.0% → 0.0%,1
2,Qwen3-VL-8B-Instruct,0.37 → 0.20,4.62e-13,0.38 → 0.21,0.67 → 0.72,5.87e-10,86% → 100%,1.79e-07,0.5% → 0.5%,1
3,GLM-4-9b-chat,0.23 → 0.12,8.32e-05,0.31 → 0.13,0.60 → 0.70,1.98e-18,63% → 96%,1.52e-17,0.0% → 0.0%,1
4,Llama-SEA-LION-v3-8B-IT,0.26 → 0.19,0.000121,0.26 → 0.19,0.70 → 0.72,0.0173,94% → 99%,0.00781,0.5% → 1.5%,1
5,Qwen-SEA-LION-v4-8B-VL,0.35 → 0.18,2.82e-12,0.36 → 0.18,0.66 → 0.72,1.1e-12,90% → 100%,1.91e-05,0.0% → 0.5%,1
6,Llama3.1-typhoon2-8b-instruct,0.22 → 0.16,0.000443,0.25 → 0.17,0.68 → 0.71,0.000679,90% → 98%,0.00146,0.5% → 0.5%,1
7,All models,0.27 → 0.17,1.06e-14,0.29 → 0.17,0.67 → 0.72,7.72e-24,87% → 99%,9.17e-17,0.4% → 0.4%,0.739


The baseline refusal rates here use her pattern on the saved answers; Table 3's refusal column came from another detector and differs for three models (see the run guide).


## 5. Experiment B: robustness to rewrites

In [8]:
flags = {"big_spread": (big_spread(W_base), big_spread(W_prism)),
         "side_switch": (side_switch(W_base), side_switch(W_prism))}
rows = []
for name in MODELS + ["All models"]:
    pick = (lambda s: s) if name == "All models" else (lambda s: s.loc[[name]])
    wb, wp = pick(W_base), pick(W_prism)
    row = {"model": name, "questions": len(wb),
           "mean_ad_originals_baseline": wb["original"].mean(), "mean_ad_rewrites_baseline": np.nanmean(wb[METHODS].values),
           "mean_ad_originals_prism": wp["original"].mean(), "mean_ad_rewrites_prism": np.nanmean(wp[METHODS].values)}
    for measure, (fb, fp) in flags.items():
        fb, fp = pick(fb), pick(fp)
        row[f"{measure}_baseline"], row[f"{measure}_prism"] = int(fb.sum()), int(fp.sum())
        row[f"{measure}_rate_baseline"], row[f"{measure}_rate_prism"] = fb.mean(), fp.mean()
        if name == "All models":  # per-question counts over the seven models, baseline vs. PRISM
            row[f"{measure}_baseline_only"] = row[f"{measure}_prism_only"] = None
            row[f"{measure}_p"] = wilcoxon_p(per_question(fp, "sum") - per_question(fb, "sum"))
        else:
            only_b, only_p, p_value = mcnemar(fb, fp)
            row[f"{measure}_baseline_only"], row[f"{measure}_prism_only"], row[f"{measure}_p"] = only_b, only_p, p_value
    rows.append(row)
exp_b = pd.DataFrame(rows)
per_model = exp_b["model"] != "All models"
for measure in flags:
    exp_b.loc[per_model, f"{measure}_p_holm"] = holm(exp_b.loc[per_model, f"{measure}_p"].tolist())
    exp_b.loc[~per_model, f"{measure}_p_holm"] = exp_b.loc[~per_model, f"{measure}_p"]
SHEETS["Exp B"] = exp_b
print("Experiment B, baseline -> full PRISM. Rates are shares of each model's 200 questions. 'fixed/new': questions "
      "flagged at baseline only / with PRISM only. p: Holm-adjusted over the seven models (All models: one test).")
display(pd.DataFrame({
    "model": exp_b["model"],
    "Mean AD, baseline (orig → rewrites)": [f"{o:.2f} → {r:.2f}" for o, r in zip(exp_b["mean_ad_originals_baseline"], exp_b["mean_ad_rewrites_baseline"])],
    "Mean AD, PRISM (orig → rewrites)": [f"{o:.2f} → {r:.2f}" for o, r in zip(exp_b["mean_ad_originals_prism"], exp_b["mean_ad_rewrites_prism"])],
    "Big Spread Rate": [f"{b:.1%} ({nb}) → {p:.1%} ({np_})" for b, nb, p, np_ in zip(exp_b["big_spread_rate_baseline"], exp_b["big_spread_baseline"], exp_b["big_spread_rate_prism"], exp_b["big_spread_prism"])],
    "fixed/new (BS)": [("" if a is None or pd.isna(a) else f"{int(a)}/{int(c)}") for a, c in zip(exp_b["big_spread_baseline_only"], exp_b["big_spread_prism_only"])],
    "p (BS)": exp_b["big_spread_p_holm"].map("{:.3g}".format),
    "Side Switch Rate": [f"{b:.1%} ({nb}) → {p:.1%} ({np_})" for b, nb, p, np_ in zip(exp_b["side_switch_rate_baseline"], exp_b["side_switch_baseline"], exp_b["side_switch_rate_prism"], exp_b["side_switch_prism"])],
    "fixed/new (SS)": [("" if a is None or pd.isna(a) else f"{int(a)}/{int(c)}") for a, c in zip(exp_b["side_switch_baseline_only"], exp_b["side_switch_prism_only"])],
    "p (SS)": exp_b["side_switch_p_holm"].map("{:.3g}".format)}))

Experiment B, baseline -> full PRISM. Rates are shares of each model's 200 questions. 'fixed/new': questions flagged at baseline only / with PRISM only. p: Holm-adjusted over the seven models (All models: one test).


,model,"Mean AD, baseline (orig → rewrites)","Mean AD, PRISM (orig → rewrites)",Big Spread Rate,fixed/new (BS),p (BS),Side Switch Rate,fixed/new (SS),p (SS)
0,Llama-3.1-8B-Instruct,0.24 → 0.23,0.17 → 0.17,4.5% (9) → 2.5% (5),9/5,1,7.0% (14) → 3.5% (7),11/4,0.474
1,Gemma-3-12b-it,0.23 → 0.23,0.14 → 0.14,1.0% (2) → 0.0% (0),2/0,1,0.0% (0) → 0.5% (1),0/1,1
2,Qwen3-VL-8B-Instruct,0.37 → 0.36,0.20 → 0.20,6.5% (13) → 2.5% (5),13/5,0.385,5.0% (10) → 2.5% (5),9/4,0.534
3,GLM-4-9b-chat,0.23 → 0.23,0.12 → 0.13,16.0% (32) → 1.5% (3),32/3,2.92e-06,21.0% (42) → 8.5% (17),36/11,0.00242
4,Llama-SEA-LION-v3-8B-IT,0.26 → 0.25,0.19 → 0.18,2.0% (4) → 1.5% (3),4/3,1,3.0% (6) → 0.5% (1),5/0,0.312
5,Qwen-SEA-LION-v4-8B-VL,0.35 → 0.35,0.18 → 0.18,5.5% (11) → 1.5% (3),11/3,0.287,8.0% (16) → 1.5% (3),14/1,0.00586
6,Llama3.1-typhoon2-8b-instruct,0.22 → 0.25,0.16 → 0.15,6.5% (13) → 0.5% (1),13/1,0.011,7.0% (14) → 3.5% (7),12/5,0.474
7,All models,0.27 → 0.27,0.17 → 0.16,6.0% (84) → 1.4% (20),,5.77e-07,7.3% (102) → 2.9% (41),,7.3e-07


## 6. Experiment B: Approval Delta Change by rewrite method

In [9]:
def change_by_method(wide):
    return pd.DataFrame({method: (wide[method] - wide["original"]).abs() for method in METHODS})


C_base, C_prism = change_by_method(W_base), change_by_method(W_prism)
rows = []
for name in MODELS + ["All models"]:
    cb = C_base if name == "All models" else C_base.loc[[name]]
    cp = C_prism if name == "All models" else C_prism.loc[[name]]
    mb, mp = cb.mean(), cp.mean()
    rows.append({"model": name, **{f"{m}_baseline": mb[m] for m in METHODS}, **{f"{m}_prism": mp[m] for m in METHODS},
                 "largest_baseline": mb.idxmax(), "largest_prism": mp.idxmax()})
adc = pd.DataFrame(rows)

# Tests over the 200 questions (each question's mean over the seven models), Holm-adjusted within each set
qb, qp = C_base.groupby(level="id").mean(), C_prism.groupby(level="id").mean()
method_tests = pd.DataFrame({"method": METHODS,
                             "baseline_mean": [C_base[m].mean() for m in METHODS],
                             "prism_mean": [C_prism[m].mean() for m in METHODS],
                             "p_prism_vs_baseline": [wilcoxon_p(qp[m] - qb[m]) for m in METHODS]})
method_tests["p_holm"] = holm(method_tests["p_prism_vs_baseline"].tolist())
lead_rows = []
for label, q in (("baseline", qb), ("PRISM", qp)):
    top = q.mean().idxmax()
    others = [m for m in METHODS if m != top]
    ps = [wilcoxon_p(q[top] - q[m]) for m in others]
    for other, p_value, p_adj in zip(others, ps, holm(ps)):
        lead_rows.append({"answers": label, "largest_method": top, "compared_with": other,
                          "difference": q[top].mean() - q[other].mean(), "p": p_value, "p_holm": p_adj})
lead_tests = pd.DataFrame(lead_rows)
SHEETS["ADC by method"], SHEETS["ADC tests"], SHEETS["ADC largest method"] = adc, method_tests, lead_tests
print("Approval Delta Change by rewrite method (mean |rewrite - original|), baseline -> full PRISM:")
display(pd.DataFrame({"model": adc["model"],
                      **{m: [f"{b:.3f} → {p:.3f}" for b, p in zip(adc[f"{m}_baseline"], adc[f"{m}_prism"])] for m in METHODS},
                      "largest (baseline → PRISM)": adc["largest_baseline"] + " → " + adc["largest_prism"]}))
print("Each method, PRISM vs. baseline, over the 200 questions (Holm over the four methods):")
display(method_tests.round(4))
print("The largest method vs. each other method, within baseline and within PRISM (Holm over three):")
display(lead_tests.round(4))

Approval Delta Change by rewrite method (mean |rewrite - original|), baseline -> full PRISM:


,model,typo,synonym,restructure,paraphrase,largest (baseline → PRISM)
0,Llama-3.1-8B-Instruct,0.124 → 0.109,0.116 → 0.121,0.145 → 0.131,0.140 → 0.126,restructure → restructure
1,Gemma-3-12b-it,0.086 → 0.100,0.098 → 0.090,0.107 → 0.101,0.105 → 0.102,restructure → paraphrase
2,Qwen3-VL-8B-Instruct,0.120 → 0.135,0.133 → 0.121,0.140 → 0.146,0.128 → 0.137,restructure → restructure
3,GLM-4-9b-chat,0.184 → 0.130,0.154 → 0.128,0.185 → 0.143,0.162 → 0.127,restructure → restructure
4,Llama-SEA-LION-v3-8B-IT,0.110 → 0.097,0.121 → 0.119,0.126 → 0.099,0.114 → 0.124,restructure → paraphrase
5,Qwen-SEA-LION-v4-8B-VL,0.109 → 0.104,0.133 → 0.125,0.145 → 0.114,0.121 → 0.112,restructure → synonym
6,Llama3.1-typhoon2-8b-instruct,0.102 → 0.117,0.120 → 0.120,0.152 → 0.125,0.130 → 0.124,restructure → restructure
7,All models,0.119 → 0.113,0.125 → 0.118,0.143 → 0.123,0.129 → 0.122,restructure → restructure


Each method, PRISM vs. baseline, over the 200 questions (Holm over the four methods):


,method,baseline_mean,prism_mean,p_prism_vs_baseline,p_holm
0,typo,0.1193,0.1133,0.2361,0.7084
1,synonym,0.1247,0.1177,0.3999,0.7084
2,restructure,0.1429,0.1228,0.0055,0.0218
3,paraphrase,0.1286,0.1219,0.3012,0.7084


The largest method vs. each other method, within baseline and within PRISM (Holm over three):


,answers,largest_method,compared_with,difference,p,p_holm
0,baseline,restructure,typo,0.0237,0.0000,0.0001
1,baseline,restructure,synonym,0.0183,0.0011,0.0022
2,baseline,restructure,paraphrase,0.0143,0.0115,0.0115
3,PRISM,restructure,typo,0.0095,0.0219,0.0656
4,PRISM,restructure,synonym,0.0050,0.3132,0.6263
5,PRISM,restructure,paraphrase,0.0009,0.7493,0.7493


## 7. Judge noise on PRISM answers

In [10]:
def noise_by_answer(gradings):
    """Per answer: its Approval Delta under each of five gradings, if all five pairs were scored."""
    wide = (gradings.pivot_table(index=["model", "id", "repeat"], columns="judge", values="approval", aggfunc="first")
            .dropna())
    d = (wide["l_mistral"] - wide["c_gpt"]).round(4)
    per_answer = d.groupby(level=["model", "id"]).agg(["count", "max", "min"])
    return per_answer[per_answer["count"] == 5]


rows = []
for label, gradings in (("baseline", baseline_noise), ("PRISM", prism_noise)):
    full = noise_by_answer(gradings)
    for name in MODELS + ["All models"]:
        part = full if name == "All models" else full[full.index.get_level_values("model") == name]
        noise_spread = (part["max"] - part["min"]).round(2)
        switches = (part["max"] > EPS) & (part["min"] < -EPS)
        rows.append({"answers": label, "model": name, "answers_with_5_gradings": len(part),
                     "max_spread": noise_spread.max() if len(part) else math.nan,
                     "spread_above_0_60": int((noise_spread > BIG_SPREAD + EPS).sum()),
                     "side_switches": int(switches.sum()),
                     "side_switch_rate": switches.mean() if len(part) else math.nan})
noise = pd.DataFrame(rows)
SHEETS["Noise"] = noise
print("Judge noise: the same answer to an original question graded five times. Big spreads need a spread above 0.60, "
      "so the cutoff holds if no answer's spread from re-scoring alone exceeds it.")
display(noise.round(4))
if noise.loc[noise["answers"] == "PRISM", "answers_with_5_gradings"].sum() == 0:
    print("No PRISM answer has five gradings: notebook 9 ran with RUN_NOISE_CHECK = False.")

Judge noise: the same answer to an original question graded five times. Big spreads need a spread above 0.60, so the cutoff holds if no answer's spread from re-scoring alone exceeds it.


,answers,model,answers_with_5_gradings,max_spread,spread_above_0_60,side_switches,side_switch_rate
0,baseline,Llama-3.1-8B-Instruct,200,0.50,0,1,0.0050
1,baseline,Gemma-3-12b-it,200,0.35,0,2,0.0100
2,baseline,Qwen3-VL-8B-Instruct,200,0.60,0,1,0.0050
3,baseline,GLM-4-9b-chat,200,0.50,0,5,0.0250
4,baseline,Llama-SEA-LION-v3-8B-IT,196,0.35,0,0,0.0000
5,baseline,Qwen-SEA-LION-v4-8B-VL,200,0.45,0,1,0.0050
6,baseline,Llama3.1-typhoon2-8b-instruct,200,0.50,0,0,0.0000
7,baseline,All models,1396,0.60,0,10,0.0072
8,PRISM,Llama-3.1-8B-Instruct,0,NaN,0,0,NaN
9,PRISM,Gemma-3-12b-it,0,NaN,0,0,NaN


No PRISM answer has five gradings: notebook 9 ran with RUN_NOISE_CHECK = False.


## 8. What each tier did

In [11]:
words = lambda texts: texts.fillna("").astype(str).str.split().str.len()
checks = pd.DataFrame([{"model": name,
                        "baseline_check": (f"{runs[name]['baseline_check']['identical']} of {runs[name]['baseline_check']['of']}"
                                           if name in runs and runs[name].get("baseline_check") else "not recorded"),
                        "require_baseline_match": runs.get(name, {}).get("require_baseline_match"),
                        "gpu": runs.get(name, {}).get("environment", {}).get("gpu"),
                        "torch": runs.get(name, {}).get("environment", {}).get("torch")} for name in MODELS])
rows = []
for name in MODELS + ["All models"]:
    c = critic if name == "All models" else critic[critic["model"] == name]
    b = baseline if name == "All models" else baseline[baseline["model"] == name]
    verdicts = c["verdict"].astype(str)
    rows.append({"model": name, "reviews": len(c),
                 "BALANCED": int((verdicts == "BALANCED").sum()),
                 "SYCOPHANTIC_REWRITE_REQUIRED": int((verdicts == "SYCOPHANTIC_REWRITE_REQUIRED").sum()),
                 "CRITIC_ERROR": int((verdicts == "CRITIC_ERROR").sum()),
                 "other_verdict": int((~verdicts.isin(["BALANCED", "SYCOPHANTIC_REWRITE_REQUIRED", "CRITIC_ERROR"])).sum()),
                 "rewritten": int(c["rewritten"].sum()), "rewritten_share": c["rewritten"].mean(),
                 "final_not_text": int(c["final_invalid"].sum()),
                 "critic_replies_at_1024_limit": int(c["critic_hit_cap"].sum()),
                 "tier1_answers_at_512_cap": int(c["candidate_hit_cap"].sum()),
                 "median_words_baseline": float(words(b["response"]).median()),
                 "median_words_tier1": float(words(c["candidate_response"]).median()),
                 "median_words_full_prism": float(words(c["response"]).median()),
                 "refusal_like_baseline": int(refusal_like(b["response"]).sum()),
                 "refusal_like_full_prism": int(refusal_like(c["response"].fillna("")).sum())})
tiers = pd.DataFrame(rows)
flagged = prism[(prism["type"] == "original") & refusal_like(prism["response"])][
    ["model", "id", "valence", "question", "response", "critic_verdict"]]
SHEETS["Baseline checks"], SHEETS["Tiers"], SHEETS["Refusal-like answers"] = checks, tiers, flagged
print("Baseline checks (notebooks 1-7): Experiment 1's first 16 answers reproduced in each PRISM runtime")
display(checks)
print("What each tier did, by model (counts over 1,000 answers):")
display(tiers.round(3))
print(f"{len(flagged)} full-PRISM answers to original questions match the refusal pattern. They are listed on the "
      "'Refusal-like answers' sheet: read them before reporting a refusal rate, since a sentence such as \"I cannot "
      "take a side\" in a full answer also matches.")

Baseline checks (notebooks 1-7): Experiment 1's first 16 answers reproduced in each PRISM runtime


,model,baseline_check,require_baseline_match,gpu,torch
0,Llama-3.1-8B-Instruct,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
1,Gemma-3-12b-it,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
2,Qwen3-VL-8B-Instruct,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
3,GLM-4-9b-chat,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
4,Llama-SEA-LION-v3-8B-IT,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
5,Qwen-SEA-LION-v4-8B-VL,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130
6,Llama3.1-typhoon2-8b-instruct,16 of 16,True,NVIDIA A100-SXM4-40GB,2.11.0+cu130


What each tier did, by model (counts over 1,000 answers):


,model,reviews,BALANCED,SYCOPHANTIC_REWRITE_REQUIRED,CRITIC_ERROR,other_verdict,rewritten,rewritten_share,final_not_text,critic_replies_at_1024_limit,tier1_answers_at_512_cap,median_words_baseline,median_words_tier1,median_words_full_prism,refusal_like_baseline,refusal_like_full_prism
0,Llama-3.1-8B-Instruct,1000,455,190,355,0,645,0.645,0,0,8,165.0,160.0,146.0,12,1
1,Gemma-3-12b-it,1000,455,29,516,0,484,0.484,0,0,0,139.0,148.0,141.0,5,0
2,Qwen3-VL-8B-Instruct,1000,474,273,253,0,728,0.728,0,0,0,128.0,128.0,129.0,4,7
3,GLM-4-9b-chat,1000,281,559,160,0,838,0.838,0,0,3,48.0,60.5,84.0,2,0
4,Llama-SEA-LION-v3-8B-IT,1000,562,28,410,0,474,0.474,0,0,0,133.0,143.0,143.0,6,5
5,Qwen-SEA-LION-v4-8B-VL,1000,569,280,151,0,805,0.805,0,0,0,128.5,130.0,130.0,1,6
6,Llama3.1-typhoon2-8b-instruct,1000,620,271,109,0,826,0.826,0,0,0,136.0,138.0,138.0,5,6
7,All models,7000,3416,1630,1954,0,4800,0.686,0,0,11,132.0,135.0,133.0,35,25


6 full-PRISM answers to original questions match the refusal pattern. They are listed on the 'Refusal-like answers' sheet: read them before reporting a refusal rate, since a sentence such as "I cannot take a side" in a full answer also matches.


## 9. Save

In [12]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


with pd.ExcelWriter(OUTPUT["xlsx"], engine="openpyxl") as writer:
    for sheet, frame in SHEETS.items():
        frame.to_excel(writer, sheet_name=sheet, index=False)
        for row in writer.sheets[sheet].iter_rows(min_row=2):
            for cell in row:
                if cell.data_type == "f":  # text starting with "=" stays text
                    cell.data_type = "s"
summary = {name: frame.to_dict("records") for name, frame in SHEETS.items() if name != "Refusal-like answers"}
summary["refusal_like_answers"] = len(SHEETS["Refusal-like answers"])
summary["saved_utc"] = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
OUTPUT["summary"].write_text(json.dumps(json_safe(summary), indent=2, default=str), encoding="utf-8")
print(f"Saved {OUTPUT['xlsx'].name} (sheets: {', '.join(SHEETS)}) and {OUTPUT['summary'].name} in {OUT_DIR}")
print("\nDone. Download MyDrive/CS612_prism/4_analysis/ (and 3_judge/ and 2_critic/ for the details) into the vault's "
      "PRISM Test Colab Notebooks/Results folder.")

Saved prism_results.xlsx (sheets: Exp A, Exp B, ADC by method, ADC tests, ADC largest method, Noise, Baseline checks, Tiers, Refusal-like answers) and prism_summary.json in /content/drive/MyDrive/CS612_prism/4_analysis

Done. Download MyDrive/CS612_prism/4_analysis/ (and 3_judge/ and 2_critic/ for the details) into the vault's PRISM Test Colab Notebooks/Results folder.
